# 1. Estudo do Perfil dos Alunos da Graduação do Instituto Federal do Espírito Santo

# 2. Membros:
Daniel Canton 2024014172, Robson Faria 2024013958 e Thiago Henrique 2024014180

# 3. Descrição dos dados

URL do Conjunto de Dados
O conjunto de dados original, contendo informações sobre os alunos de graduação do IFES, foi obtido a partir do Portal de Dados Abertos do Governo Federal, através da seguinte URL: https://dados.gov.br/dados/conjuntos-dados/alunos-da-graduacao

# Domínio
O domínio de origem dos dados é o dados.gov.br. Este é o portal oficial de dados abertos do Governo Federal brasileiro, que centraliza e disponibiliza conjuntos de dados de diversas áreas da administração pública para consulta e utilização pela sociedade.

# Processamento dos Dados
Os dados foram originalmente disponibilizados em um único arquivo no formato CSV. Este formato apresentava um esquema relacional não normalizado, com significativa redundância de informações e suscetível a anomalias de inserção, atualização e deleção.

Para estruturar um banco de dados relacional íntegro e eficiente, foi realizado um processo de normalização para a Terceira Forma Normal (3FN). Os seguintes passos foram executados:

Análise e Decomposição: O esquema inicial foi decomposto em cinco tabelas distintas para representar as entidades principais e seus relacionamentos: Instituicao, Departamento, Curso, Aluno e Inscricao.

Criação e População de Entidades:

Dados únicos de instituições e cursos foram extraídos do arquivo original para popular as tabelas Instituicao e Curso. Chaves primárias numéricas (id_instituicao, id_curso) foram geradas para cada registro.

A tabela Aluno foi populada com os dados únicos de cada estudante, utilizando a matricula como chave primária.

Tratamento de Dados e Enriquecimento do Modelo:

Para a entidade Departamento, cujos dados não estavam explícitos no arquivo original, foram criados registros fictícios (ex: "Departamento de Engenharias", "Ciências Exatas e da Natureza") para permitir a categorização dos cursos e cumprir os requisitos do projeto.

Para atender à especificação do trabalho de que cada entidade deveria possuir ao menos dois atributos além do identificador, as entidades 

Instituicao e Departamento foram enriquecidas com os atributos telefone e chefe_departamento, respectivamente.

Criação da Tabela Associativa: A tabela Inscricao foi populada iterando sobre cada linha do CSV original. Para cada registro, as chaves primárias correspondentes das tabelas Aluno, Curso e Instituicao foram recuperadas e inseridas como chaves estrangeiras, estabelecendo assim os relacionamentos corretos e resolvendo a relação M:N entre alunos e cursos.

Ao final deste processo, os dados originalmente contidos em um único arquivo foram reestruturados em um banco de dados relacional normalizado, pronto para a execução de consultas analíticas.


In [ ]:
import pandas as pd
import sqlite3

#Fase 1: Preparação do Ambiente e Criação das Tabelas

# 1.1 Carregar os dados do CSV para um DataFrame do pandas
df = pd.read_csv('BD_TP2.csv')

# 1.2 Conectar ao banco de dados (ou criar se não existir)

# 1.3 deleta o banco caso já exista
import os
if os.path.exists("dados_academicos.db"):
    os.remove("dados_academicos.db")
    
# 1.4 cria o banco
conn = sqlite3.connect('dados_academicos.db')
cursor = conn.cursor()
# --- Criar as tabelas do esquema normalizado ---

# 1.5 Tabela Instituicao
cursor.execute('''
CREATE TABLE IF NOT EXISTS Instituicao (
    id_instituicao INTEGER PRIMARY KEY AUTOINCREMENT,
    campus TEXT UNIQUE NOT NULL,
    telefone TEXT
);
''')

# 1.6 Tabela Departamento
cursor.execute('''
CREATE TABLE IF NOT EXISTS Departamento (
    id_departamento INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT UNIQUE NOT NULL,
    chefe_departamento TEXT
);
''')

# 1.7 Tabela Curso
cursor.execute('''
CREATE TABLE IF NOT EXISTS Curso (
    id_curso INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    turno TEXT NOT NULL,
    id_departamento INTEGER,
    FOREIGN KEY (id_departamento) REFERENCES Departamento(id_departamento)
);
''')

# 1.8 Tabela Aluno
cursor.execute('''
CREATE TABLE IF NOT EXISTS Aluno (
    matricula TEXT PRIMARY KEY,
    nome TEXT,
    sexo TEXT,
    periodo_inicial TEXT
);
''')

# 1.9 Tabela Associativa Inscricao
cursor.execute('''
CREATE TABLE IF NOT EXISTS Inscricao (
    matricula_aluno TEXT,
    id_curso INTEGER,
    id_instituicao INTEGER,
    turma TEXT,
    PRIMARY KEY (matricula_aluno, id_curso),
    FOREIGN KEY (matricula_aluno) REFERENCES Aluno(matricula),
    FOREIGN KEY (id_curso) REFERENCES Curso(id_curso),
    FOREIGN KEY (id_instituicao) REFERENCES Instituicao(id_instituicao)
);
''')

print("Tabelas criadas com sucesso.")

#Fase 2: Extrair e Popular as Tabelas de Entidades
# 2.1 Desativar a checagem de chaves estrangeiras para inserção em massa
cursor.execute('PRAGMA foreign_keys = OFF;')

# 2.2 Popular a tabela Instituicao
# 2.2.1 Gerar telefones fixos fictícios baseados no modelo do ES
import random

def gerar_telefone_fixo_es():
    ddd = random.choice(['27','28'])
    numero = '3' + ''.join(str(random.randint(0,9)) for _ in range(7))
    return f"({ddd}) {numero[:4]}-{numero[4:]}"

# 2.2.2 Popular tabela
instituicoes_unicas = df['nome_instituicao'].unique()
instituicao_map = {}
for i, campus_nome in enumerate(instituicoes_unicas):
    # 2.2.1 Inserir no DB e guardar o ID gerado no mapa
    telefone_fixo = gerar_telefone_fixo_es()
    cursor.execute("INSERT INTO Instituicao (campus, telefone) VALUES (?, ?)", (campus_nome, telefone_fixo))
    instituicao_map[campus_nome] = cursor.lastrowid
print("Tabela Instituicao populada.")


# 2.3 Popular a tabela Departamento (com dados fictícios, pois os originais estão vazios)
departamentos_ficticios = {
    "Engenharias e Tecnologia": "Maria Silva",
    "Ciências Exatas e da Natureza": "João Souza",
    "Ciências Humanas e Sociais": "Ana Costa",
    "Ciências Agrárias": "Carlos Pereira"
}
departamento_map = {}
for nome, chefe in departamentos_ficticios.items():
    cursor.execute("INSERT INTO Departamento (nome, chefe_departamento) VALUES (?, ?)", (nome, chefe))
    departamento_map[nome] = cursor.lastrowid
print("Tabela Departamento populada.")


# 2.4 Popular a tabela Curso
cursos_unicos = df[['nome_curso', 'turno_curso']].drop_duplicates()
curso_map = {}
for index, row in cursos_unicos.iterrows():
    nome_curso = row['nome_curso']
    turno_curso = row['turno_curso']
    
    # 2.4.1 Lógica para associar um curso a um departamento fictício
    id_depto = None
    if 'Engenharia' in nome_curso or 'Sistemas' in nome_curso or 'Tecnologia' in nome_curso:
        id_depto = departamento_map["Engenharias e Tecnologia"]
    elif 'Química' in nome_curso or 'Física' in nome_curso or 'Matemática' in nome_curso:
        id_depto = departamento_map["Ciências Exatas e da Natureza"]
    elif 'Administração' in nome_curso or 'Pedagogia' in nome_curso or 'Letras' in nome_curso or 'Geografia' in nome_curso:
        id_depto = departamento_map["Ciências Humanas e Sociais"]
    elif 'Agronomia' in nome_curso or 'Zootecnia' in nome_curso or 'Cafeicultura' in nome_curso or 'Aquicultura' in nome_curso:
        id_depto = departamento_map["Ciências Agrárias"]
        
    cursor.execute("INSERT INTO Curso (nome, turno, id_departamento) VALUES (?, ?, ?)", (nome_curso, turno_curso, id_depto))
    curso_map[(nome_curso, turno_curso)] = cursor.lastrowid
print("Tabela Curso populada.")


# 2.5 Popular a tabela Aluno
# 2.5.1 Lógica para criar nomes ficticios
from faker import Faker

faker = Faker('pt_BR')

nomes_usados = set()

def gerar_nome_unico(sexo):
    while True:
        if sexo == 'M':
            nome = faker.first_name_male() + ' ' + faker.last_name()
        else:
            nome = faker.first_name_female() + ' ' + faker.last_name()
        if nome not in nomes_usados:
            nomes_usados.add(nome)
            return nome

alunos_unicos = df.drop_duplicates(subset=['matricula_aluno'])

# 2.5.2 Lógica para popular com dados da tabela e nomes fictícios
for index, row in alunos_unicos.iterrows():
    sexo = row['sexo_aluno']
    nome_ficticio = gerar_nome_unico(sexo)
    
    cursor.execute("""
        INSERT INTO Aluno (matricula, nome, sexo, periodo_inicial)
        VALUES (?, ?, ?, ?)
    """, (row['matricula_aluno'], nome_ficticio, sexo, row['periodo_inicial_aluno']))

print("Tabela Aluno populada")

#Fase 3: Popular a Tabela Associativa Inscricao
# 3.1 Iterar sobre cada linha do DataFrame original para criar as inscrições
for index, row in df.iterrows():
    # 3.1.1 Obter os IDs das chaves estrangeiras usando os mapas
    matricula = row['matricula_aluno']
    id_curso = curso_map.get((row['nome_curso'], row['turno_curso']))
    id_instituicao = instituicao_map.get(row['nome_instituicao'])
    turma = row['turma_aluno']
    
    # 3.1.2 Inserir o registro de inscrição
    if id_curso and id_instituicao: # Garante que os dados existem
        cursor.execute("INSERT OR IGNORE INTO Inscricao (matricula_aluno, id_curso, id_instituicao, turma) VALUES (?, ?, ?, ?)",
                       (matricula, id_curso, id_instituicao, turma))

# 3.2 Reativar a checagem de chaves estrangeiras
cursor.execute('PRAGMA foreign_keys = ON;')
print("Tabela Inscricao populada.")

# 3.3 Commit e close no banco para possibilitar futuras alterações
# conn.commit() 
# conn.close()

Tabelas criadas com sucesso.
Tabela Instituicao populada.
Tabela Departamento populada.
Tabela Curso populada.
Tabela Aluno populada
Tabela Inscricao populada.


In [2]:
pd.read_sql_query("SELECT * FROM Curso;", conn)

,id_curso,nome,turno,id_departamento
0,1,Bacharelado em Química Industrial,Integral,2.0
1,2,Engenharia Mecânica,Integral,1.0
2,3,Licenciatura em Química,Noturno,2.0
3,4,Bacharelado em Química Industrial,Noturno,2.0
4,5,Licenciatura em Química,Integral,2.0
...,...,...,...,...
85,86,Engenharia Civil,Matutino,1.0
86,87,Licenciatura em Letras,Integral,3.0
87,88,Licenciatura em Matemática - Noturno,Integral,2.0
88,89,Licenciatura em Letras,Matutino,3.0


# 4. Diagrama ER <img src="diagrama_er.jpg" alt="Diagrama ER" width="800"/>

# 5. Diagrama relacional <img src="diagrama_relacional.jpg" alt="Diagrama Relacional" width="1000"/>

# 6. Consultas

## 6.1 Duas consultas envolvendo seleção e projeção

### 6.1.1 Consulta 1

In [5]:
cursor.execute('''
    SELECT *
    FROM Aluno
    WHERE periodo_inicial >= '2024/1' AND sexo = 'F'
''')
resultado = cursor.fetchall()
print(resultado)

[('20241BAQUIN0127', 'Ana Clara Leão', 'F', '2024/1'), ('20241BAQUIN0011', 'Catarina Camargo', 'F', '2024/1'), ('20241BAQUIN0097', 'Melissa Freitas', 'F', '2024/1'), ('20241LQUIM0113', 'Luiza Nunes', 'F', '2024/1'), ('20251LQUIM0170', 'Gabriela Machado', 'F', '2025/1'), ('20251BAQUIN0190', 'Maria Fernanda Nunes', 'F', '2025/1'), ('20242LQUIM0014', 'Rafaela Garcia', 'F', '2024/2'), ('20251BAQUIN0220', 'Pietra da Paz', 'F', '2025/1'), ('20251BAQUIN0204', 'Kamilly Abreu', 'F', '2025/1'), ('20251LQUIM0218', 'Ana Júlia Alves', 'F', '2025/1'), ('20241ENG.MEC0059', 'Laís Rocha', 'F', '2024/1'), ('20241ENG.MEC0458', 'Stephany Câmara', 'F', '2024/1'), ('20251LQUIM0030', 'Elisa Sales', 'F', '2025/1'), ('20251LQUIM0196', 'Liz Costa', 'F', '2025/1'), ('20251BAQUIN0255', 'Maria Alice Siqueira', 'F', '2025/1'), ('20251LQUIM0048', 'Letícia Peixoto', 'F', '2025/1'), ('20241LQUIM0083', 'Rebeca Azevedo', 'F', '2024/1'), ('20241ENG.MEC0180', 'Emanuella Costa', 'F', '2024/1'), ('20251BAQUIN0034', 'Maria E

### 6.1.2 Consulta 2

In [6]:
cursor.execute('''
    SELECT matricula, nome, periodo_inicial
    FROM Aluno
    WHERE periodo_inicial < '2020/1'
''')
resultado = cursor.fetchall()
print(resultado)

[('20171BAQUIN0384', 'Caroline Cavalcante', '2017/1'), ('20181ENG.MEC0250', 'Danilo Ferreira', '2018/1'), ('20191ENG.MEC0078', 'Antônio Lopes', '2019/1'), ('20181ENG.MEC0510', 'Isaque Nascimento', '2018/1'), ('20191ENG.MEC0418', 'João Felipe Nogueira', '2019/1'), ('20181ENG.MEC0285', 'Renan Pinto', '2018/1'), ('20171LQUIM0275', 'Stephany Sampaio', '2017/1'), ('20161LQUIM0090', 'Lorena Ferreira', '2016/1'), ('20181LQUIM0140', 'Eduardo da Mota', '2018/1'), ('20171ENG.MEC0450', 'Vitor Ramos', '2017/1'), ('20181LQUIM0345', 'Amanda Silva', '2018/1'), ('20192LQUIM0014', 'Liam da Rocha', '2019/2'), ('20191BAQUIN0360', 'Elisa Pimenta', '2019/1'), ('20191ENG.MEC0477', 'Calebe da Costa', '2019/1'), ('20191ENG.MEC0140', 'Diego Gonçalves', '2019/1'), ('20161ENG.MEC0098', 'Theodoro Caldeira', '2016/1'), ('20181ENG.MEC0056', 'Renan Sousa', '2018/1'), ('20181LQUIM0205', 'Otávio Mendes', '2018/1'), ('20181LQUIM0221', 'Luísa Nascimento', '2018/1'), ('20191LQUIM0393', 'Gael Rodrigues', '2019/1'), ('2019

## 6.2 Três consultas envolvendo junção de duas relações

### 6.2.1 Consulta 3

In [7]:
cursor.execute('''
    SELECT a.nome AS nome_aluno, c.nome AS nome_curso
    FROM Aluno a
    JOIN INSCRICAO i ON a.matricula = i.matricula_aluno
    JOIN CURSO c ON i.id_curso = c.id_curso
''')
resultado = cursor.fetchall()
print(resultado)

[('Benício Ramos', 'Engenharia de Controle e Automação'), ('Danilo Teixeira', 'Engenharia de Controle e Automação'), ('Ryan Pastor', 'Engenharia de Controle e Automação'), ('Mariah Sousa', 'Engenharia Sanitária e Ambiental'), ('Benjamin Martins', 'Engenharia de Controle e Automação'), ('Benício Mendonça', 'Engenharia de Controle e Automação'), ('André Nascimento', 'Engenharia de Produção'), ('Liam Macedo', 'Licenciatura em Química'), ('Miguel da Paz', 'Engenharia de Controle e Automação'), ('Nicolas Castro', 'Engenharia de Controle e Automação'), ('Clara da Conceição', 'Licenciatura em Química'), ('Josué Macedo', 'Engenharia Metalúrgica - CH Comp 200'), ('Marina Caldeira', 'Engenharia Metalúrgica - CH Comp 200'), ('Luiz Henrique Costa', 'Sistemas de Informação'), ('Luiz Otávio Melo', 'Engenharia de Controle e Automação'), ('Allana Rodrigues', 'Engenharia Elétrica'), ('Maria Correia', 'Licenciatura em Química'), ('Melina Albuquerque', 'Sistemas de Informação'), ('João Miguel Barros', 'E

### 6.2.2 Consulta 4

In [8]:
cursor.execute('''
    SELECT ins.id_instituicao, inst.campus, COUNT(DISTINCT ins.matricula_aluno) AS total_alunos
    FROM Inscricao ins
    JOIN Instituicao inst ON ins.id_instituicao = inst.id_instituicao
    GROUP BY ins.id_instituicao, inst.campus
''')
resultado = cursor.fetchall()
print(resultado)

[(1, 'Campus Aracruz', 358), (2, 'Campus Barra de São Francisco', 110), (3, 'Campus Cachoeiro de Itapemirim', 540), (4, 'Campus Cariacica', 486), (5, 'Campus Centro-Serrano', 105), (6, 'Campus Colatina', 562), (7, 'Campus de Alegre', 280), (8, 'Campus Guarapari', 464), (9, 'Campus Ibatiba', 167), (10, 'Campus Itapina', 457), (11, 'Campus Linhares', 249), (12, 'Campus Montanha', 50), (13, 'Campus Nova Venécia', 236), (14, 'Campus Piúma', 101), (15, 'Campus Santa Teresa', 334), (16, 'Campus São Mateus', 268), (17, 'Campus Serra', 1179), (18, 'Campus Venda Nova do Imigrante', 265), (19, 'Campus Viana', 109), (20, 'Campus Vila Velha', 650), (21, 'Campus Vitória', 1915)]


### 6.2.3 Consulta 5

In [9]:
cursor.execute('''
    SELECT d.nome AS departamento, COUNT(DISTINCT i.matricula_aluno) AS total_alunos
    FROM Departamento d
    JOIN Curso c ON d.id_departamento = c.id_departamento
    JOIN Inscricao i ON c.id_curso = i.id_curso
    GROUP BY d.nome
''')
resultado = cursor.fetchall()
print(resultado)

[('Ciências Agrárias', 425), ('Ciências Exatas e da Natureza', 798), ('Ciências Humanas e Sociais', 2282), ('Engenharias e Tecnologia', 4220)]


## 6.3 Três consultas envolvendo junção de três ou mais relações

### 6.3.1 Consulta 6

In [10]:
cursor.execute('''
    SELECT c.nome AS curso, c.turno, i.campus
    FROM Curso c
    JOIN Inscricao ins ON c.id_curso = ins.id_curso
    JOIN Instituicao i ON ins.id_instituicao = i.id_instituicao
    WHERE c.turno = 'Noturno'
    GROUP BY c.nome, c.turno, i.campus
''')
resultado = cursor.fetchall()
print(resultado)

[('Bacharelado em Administração', 'Noturno', 'Campus Colatina'), ('Bacharelado em Administração', 'Noturno', 'Campus Venda Nova do Imigrante'), ('Bacharelado em Administração - Campus Guarapari', 'Noturno', 'Campus Guarapari'), ('Bacharelado em Administração - Centro-Serrano', 'Noturno', 'Campus Centro-Serrano'), ('Bacharelado em Administração - campus Barra de São Francisco', 'Noturno', 'Campus Barra de São Francisco'), ('Bacharelado em Administração Campus Linhares', 'Noturno', 'Campus Linhares'), ('Bacharelado em Ciência e Tecnologia de Alimentos', 'Noturno', 'Campus Venda Nova do Imigrante'), ('Bacharelado em Ciências Econômicas', 'Noturno', 'Campus Cariacica'), ('Bacharelado em Química Industrial', 'Noturno', 'Campus Aracruz'), ('Ciências Biológicas', 'Noturno', 'Campus Santa Teresa'), ('Engenharia Mecânica', 'Noturno', 'Campus Aracruz'), ('Engenharia de Controle e Automação', 'Noturno', 'Campus Serra'), ('Engenharia de Produção', 'Noturno', 'Campus Cariacica'), ('Licenciatura em 

### 6.3.2 Consulta 7

In [11]:
cursor.execute('''
    SELECT DISTINCT d.nome AS departamento, c.nome AS curso, a.nome AS aluna
    FROM Departamento d
    JOIN Curso c ON c.id_departamento = d.id_departamento
    JOIN Inscricao ins ON c.id_curso = ins.id_curso
    JOIN Aluno a ON ins.matricula_aluno = a.matricula
    WHERE a.sexo = 'F'
''')
resultado = cursor.fetchall()
print(resultado)

[('Engenharias e Tecnologia', 'Engenharia Sanitária e Ambiental', 'Mariah Sousa'), ('Ciências Exatas e da Natureza', 'Licenciatura em Química', 'Clara da Conceição'), ('Engenharias e Tecnologia', 'Engenharia Metalúrgica - CH Comp 200', 'Marina Caldeira'), ('Engenharias e Tecnologia', 'Engenharia Elétrica', 'Allana Rodrigues'), ('Ciências Exatas e da Natureza', 'Licenciatura em Química', 'Maria Correia'), ('Engenharias e Tecnologia', 'Sistemas de Informação', 'Melina Albuquerque'), ('Engenharias e Tecnologia', 'Engenharia de Produção', 'Laís da Conceição'), ('Engenharias e Tecnologia', 'Engenharia Sanitária e Ambiental', 'Júlia Caldeira'), ('Engenharias e Tecnologia', 'Engenharia de Controle e Automação', 'Antonella Caldeira'), ('Engenharias e Tecnologia', 'Engenharia de Controle e Automação', 'Joana Alves'), ('Engenharias e Tecnologia', 'Engenharia de Produção', 'Giovanna Novais'), ('Engenharias e Tecnologia', 'Engenharia Metalúrgica - CH Comp 200', 'Maria Isis Rodrigues'), ('Engenhari

### 6.3.3 Consulta 8

In [12]:
cursor.execute('''
    SELECT a.nome AS aluno, c.nome AS curso, i.campus
    FROM Aluno a
    JOIN Inscricao ins ON a.matricula = ins.matricula_aluno
    JOIN Curso c ON ins.id_curso = c.id_curso
    JOIN Instituicao i ON ins.id_instituicao = i.id_instituicao
''')
resultado = cursor.fetchall()
print(resultado)

[('Luiz Gustavo Duarte', 'Bacharelado em Química Industrial', 'Campus Aracruz'), ('Davi Miguel Nogueira', 'Engenharia Mecânica', 'Campus Aracruz'), ('João Miguel Carvalho', 'Licenciatura em Química', 'Campus Aracruz'), ('Ravi Lucca Peixoto', 'Bacharelado em Química Industrial', 'Campus Aracruz'), ('Ana Clara Leão', 'Bacharelado em Química Industrial', 'Campus Aracruz'), ('Catarina Camargo', 'Bacharelado em Química Industrial', 'Campus Aracruz'), ('Zoe Caldeira', 'Licenciatura em Química', 'Campus Aracruz'), ('Mathias Dias', 'Engenharia Mecânica', 'Campus Aracruz'), ('Melissa Freitas', 'Bacharelado em Química Industrial', 'Campus Aracruz'), ('Beatriz Martins', 'Licenciatura em Química', 'Campus Aracruz'), ('Caroline Cavalcante', 'Bacharelado em Química Industrial', 'Campus Aracruz'), ('Luiza Nunes', 'Licenciatura em Química', 'Campus Aracruz'), ('Bruna Farias', 'Engenharia Mecânica', 'Campus Aracruz'), ('Maria Julia Câmara', 'Licenciatura em Química', 'Campus Aracruz'), ('Gabriela Macha

## 6.4 Duas consultas envolvendo agregação sobre junção de duas ou mais relações

### 6.4.1 Consulta 9

In [17]:
cursor.execute('''
    SELECT c.id_curso, c.nome AS curso, COUNT(*) AS total_alunos
    FROM Curso c
    JOIN Inscricao i ON c.id_curso = i.id_curso
    GROUP BY c.nome
    ORDER BY c.id_curso
''')
resultado = cursor.fetchall()
print(resultado)

[(1, 'Bacharelado em Química Industrial', 207), (2, 'Engenharia Mecânica', 567), (3, 'Licenciatura em Química', 188), (7, 'Bacharelado em Administração - campus Barra de São Francisco', 110), (8, 'Licenciatura em Matemática', 132), (9, 'Engenharia de Minas', 91), (10, 'ENGENHARIA MECÂNICA', 121), (11, 'Sistemas de Informação', 600), (12, 'Licenciatura em Informática EAD', 23), (13, 'Engenharia de Produção', 255), (15, 'Bacharelado em Ciências Econômicas', 109), (16, 'Licenciatura em Física', 106), (18, 'Bacharelado em Física', 16), (21, 'Bacharelado em Administração - Centro-Serrano', 105), (22, 'Bacharelado em Administração', 383), (25, 'Bacharelado em Sistemas de Informação', 187), (26, 'Bacharelado em Arquitetura e Urbanismo', 163), (28, 'Licenciatura em Ciências Biológicas', 93), (29, 'Superior de Tecnologia em Análise e Desenvolvimento de Sistemas', 112), (30, 'Bacharel em Engenharia de Aquicultura', 16), (31, 'Superior de Tecnologia em Cafeicultura', 40), (33, 'Bacharelado em Ciê

### 6.4.2 Consulta 10

In [19]:
cursor.execute('''
    SELECT c.turno, COUNT(*) AS total_alunos_2024_1
    FROM Aluno a
    JOIN Inscricao ins ON a.matricula = ins.matricula_aluno
    JOIN Curso c ON ins.id_curso = c.id_curso
    WHERE a.periodo_inicial >= '2024/1'
    GROUP BY c.turno
''')
resultado = cursor.fetchall()
print(resultado)

[('EAD', 411), ('Integral', 2151), ('Matutino', 120), ('Noturno', 1027), ('Não aplicável', 77)]


# 7. Autoavaliação dos membros

In [ ]:
!pip install faker